In [1]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(model_name="gpt-5-mini")

## Tools

In [2]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("What is the capital of France?")

/var/folders/hq/q_jg97r105396k0c5t15jksc0000gq/T/ipykernel_48310/1122946680.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


"31 Oct 2025 ... Paris is the capital and largest city of France, with an estimated city population of 2.04 million in an area of 105.4 km 2 (40.7 sq mi), and a metropolitan ... 13 Aug 2025 ... Paris is the capital of France, famous for its history, culture, and beauty. Known as the “City of Light,” it has landmarks like the Eiffel Tower, Louvre Museum ... 29 May 2026 ... Paris, the capital of France, is famed for its elegant architecture, world-class museums, romantic atmosphere, and iconic landmarks. From the Eiffel Tower ... 3 days ago ... Paris is the capital city of France. Key Points. France:. Capital: Paris; President: Emmanuel Macron (As of Jan 2022). Prime Minister ... 7 Apr 2026 ... @Lionfield. Subscribe. What is the capital of FRANCE? 114K. 4,614. Share. Video unavailable. This content isn't available. Skip video."

In [3]:
from  langchain_community.tools import tool

In [4]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    """
    A tool that uses DuckDuckGo to search for information.
    """
    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    response = search.invoke(query)

    return response

In [5]:
tool_duckduckgo_search.invoke("What is the capital of France?")

"31 Oct 2025 ... Paris is the capital and largest city of France, with an estimated city population of 2.04 million in an area of 105.4 km 2 (40.7 sq mi), and a metropolitan ... 13 Aug 2025 ... Paris is the capital of France, famous for its history, culture, and beauty. Known as the “City of Light,” it has landmarks like the Eiffel Tower, Louvre Museum ... 29 May 2026 ... Paris, the capital of France, is famed for its elegant architecture, world-class museums, romantic atmosphere, and iconic landmarks. From the Eiffel Tower ... 3 days ago ... Paris is the capital city of France. Key Points. France:. Capital: Paris; President: Emmanuel Macron (As of Jan 2022). Prime Minister ... 7 Apr 2026 ... @Lionfield. Subscribe. What is the capital of FRANCE? 114K. 4,614. Share. Video unavailable. This content isn't available. Skip video."

In [6]:
@tool 
def tool_wikipedia_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about persons, places, etc. """

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia.invoke(query)

    return response

In [7]:
tool_wikipedia_search.invoke("Sam Altman")

'Page: Sam Altman\nSummary: Samuel Harris Altman (born April 22, 1985) is an American entrepreneur and investor who has been the chief executive officer (CEO) of the artificial intelligence company OpenAI since 2019.\nAltman attended Stanford University for two years before he dropped out and co-founded Loopt, a smartphone geosocial networking application. Loopt was acquired by Green Dot Corporation for $43.4 million in March 2012. In 2011, Altman joined Y Combinator, a technology startup accelerator and venture capital firm, and was the company\'s president from 2014 to 2019. He is a billionaire, due to investments in over 400 companies including Reddit, Worldcoin, Helion Energy and Instacart.\nAltman co-founded OpenAI in 2015 and became its CEO in 2019, a role that made him a prominent figure of the AI boom. He supervised the launch of ChatGPT in November 2022. In 2023, he was ousted by the organization\'s board of directors for not being "consistently candid". The move was met with 

In [8]:
@tool
def tool_personal_info(name: str) -> str:
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "John Doe",
        "age": 30,
        "occupation": "Software Engineer"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "occupation": "Data Scientist"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."

In [9]:
tool_personal_info.invoke("John Doe")

'John Doe is 30 years old and works as a Software Engineer.'

## Bind Tools

In [10]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_personal_info
]

In [11]:
llm_bind = llm.bind_tools(toolkit)

In [12]:
llm_bind.invoke("What's the age of John Doe?. Make tool calls if necessary.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 246, 'total_tokens': 272, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E32wzP1TjARZ2D9vZtQSWAAEejIa6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f763b-5c35-76a0-bc7d-f9ca8eaf1dce-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'call_3yXLOYXtfanSrtREuHzxU0d6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 246, 'output_tokens': 26, 'total_tokens': 272, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 're

## ReAct Agent

In [13]:
from langchain.agents import create_agent

my_agent = create_agent(llm_bind, toolkit)

In [14]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the age of John Doe?. Make tool calls if necessary."}]}
)

{'messages': [HumanMessage(content="What's the age of John Doe?. Make tool calls if necessary.", additional_kwargs={}, response_metadata={}, id='a915465b-8a57-48a7-a9a4-da32a5940889'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 246, 'total_tokens': 272, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E32x1bi1fo9qYcyWPhZIwnWcYOkW2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f763b-6c35-7380-ae93-c88e6fabd029-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'call_6y2qBWfD9CwDmPaFCkiaAapX', 'type': 'tool_call'}], invalid_t

In [37]:
@tool
def tool_rag(query: str) -> str:
    """Use this tool when you need to answer questions based on NovaSpehere Organization's documentation.""" 

    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings
    embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
    chroma_db_con = Chroma(persist_directory="../RAG/chroma_db", embedding_function=embed_model)

    # Retrieve relevant documents from the vector store
    relevant_docs = chroma_db_con.similarity_search(query, k=2)
    relevant_docs_content = "\n".join([doc.page_content for doc in relevant_docs])
    return relevant_docs_content

In [38]:
tool_rag.invoke("When was NovaSphere Organization founded?")

'Today, NovaSphere Technologies is considered a reliable organization that provides data\nIn 2018, NovaSphere Technologies started getting more attention in the local technology'